# Data Structure
Objective: Construct schema, loaders, and primary tables for analysis

## Import

In [1]:
# Making sure all files that need to read each other are able to
import sys
from pathlib import Path

# Find repo root by walking up to the dir containing src/, so imports work no matter the launch dir
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import paths

# Libraries
import pandas as pd
import numpy as np
import duckdb

# Data paths
from paths import ONET_PATH
from paths import CPS_PATH
from paths import ACS_PATH
from paths import DUCKDB_PATH

## Construct Loaders
- Every source's loader will return the defined tables/columns/dtypes
- occ2010 is Int64 everywhere to prevent the float↔int merge bug

### Skill Importance Loader

In [2]:
# Create SKILL_COLS list and define SKILLS_PATHS constant
SKILL_COLS = [  # the 35 O*NET skill names, in matrix order
    "readingcomprehension", "activelistening", "writing", "speaking", "mathematics",
    "science", "criticalthinking", "activelearning", "learningstrategies", "monitoring",
    "socialperceptiveness", "coordination", "persuasion", "negotiation", "instructing",
    "serviceorientation", "complexproblemsolving", "operationsanalysis", "technologydesign",
    "equipmentselection", "installation", "programming", "operationsmonitoring",
    "operationandcontrol", "equipmentmaintenance", "troubleshooting", "repairing",
    "qualitycontrolanalysis", "judgmentanddecisionmaking", "systemsanalysis",
    "systemsevaluation", "timemanagement", "managementoffinancialresources",
    "managementofmaterialresources", "managementofpersonnelresources",
]

SKILLS_PATHS = {
    "onet": ONET_PATH
}

In [3]:
# Define function to load in skill importance vectors dataset from source
def load_skills(source: str) -> pd.DataFrame:
    """Occupation-level skill importance vectors

    Returns one row per occupation:
        occ2010      : Int64      # canonical key
        occ2010_title: string     # display only
        <skillname>  : float64    # 35 readable-named skill cols (readingcomprehension, ...)

    Universe: civilian occs holding a skill vector. No military / niu.
    """
    path = SKILLS_PATHS[source]                      # raises KeyError on unknown source
    df = pd.read_stata(path, convert_categoricals=False)   # convert_categoricals=False so we get codes, not labels

    df = df[["occ2010", "occ2010_title"] + SKILL_COLS].copy()

    df["occ2010"] = df["occ2010"].astype("Int64")   # contract: Int64 key everywhere
    df["occ2010_title"] = df["occ2010_title"].astype("string")
    df[SKILL_COLS] = df[SKILL_COLS].astype("float64")

    # Universe lock (Task 0): drop any occ2010 missing a code; military already absent from O*NET.
    df = df.dropna(subset=["occ2010"]).reset_index(drop=True)

    return df

In [ ]:
# # Test/Check
skill_matrix_df = load_skills("onet")
print(skill_matrix_df.shape)          # expect ~ (474, 37)
print(skill_matrix_df.head())

   occ2010                              occ2010_title  readingcomprehension  \
0       10           chief executives and legislators                  4.12   
1       20            general and operations managers                  4.00   
2       40        advertising and promotions managers                  3.75   
3       50               marketing and sales managers                  3.88   
4       60  public relations and fundraising managers                  3.88   

   activelistening   writing  speaking  mathematics  science  \
0         4.000000  4.120000  4.250000     3.250000  1.62000   
1         4.000000  3.500000  4.000000     2.620000  1.50000   
2         4.120000  3.750000  4.000000     3.000000  1.62000   
3         3.953274  3.475928  3.953274     2.902654  1.67062   
4         4.000000  3.880000  4.120000     3.000000  1.25000   

   criticalthinking  activelearning  ...  troubleshooting  repairing  \
0              4.38         3.75000  ...             1.50        1.0

### Occupation Transition Pairs


In [5]:
# Weighted median: smallest value where cumulative weight crosses half the total
def weighted_median(values: pd.Series, weights: pd.Series) -> float:
    d = pd.DataFrame({"v": values, "w": weights}).dropna().sort_values("v")
    cutoff = d["w"].sum() / 2
    return d.loc[d["w"].cumsum() >= cutoff, "v"].iloc[0] # type: ignore[union-attr]


def build_occ_level(acs: pd.DataFrame) -> pd.DataFrame:
    """Per-occupation wage + degree-screen inputs from ACS (held-constant controls).

        occ2010   : Int64
        wage      : float64   # weighted median hourw89
        ba_share  : float64   # weighted share with ed_cat==3 (BA+)
        wageg     : Int64     # 1=Low 2=Middle 3=Upper 4=High vs national median
    """
    acs = acs[(acs["in_uni"] == 1) & acs["hourw89"].notna()].copy() # load in_uni acs + no nulls

    # drop non-civilian (in_uni takes care of this but just to be safe)
    DROP = {9999, 9920, 9800, 9810, 9820, 9830}
    acs = acs[~acs["occ2010"].isin(DROP)]

    # National anchor: weighted median wage across all in-universe workers.
    nat_med = weighted_median(acs["hourw89"], acs["perwt"])

    # Per-occupation weighted median wage.
    occ = (
        acs.groupby("occ2010")
           .apply(lambda g: weighted_median(g["hourw89"], g["perwt"]))
           .reset_index(name="wage")
    )

    # Per-occupation weighted BA+ share = Σ(perwt where ed_cat==3) / Σ(perwt).
    acs["is_ba"] = (acs["ed_cat"] == 3).astype(float)
    ba = (
        acs.groupby("occ2010")
           .apply(lambda g: np.average(g["is_ba"], weights=g["perwt"]))
           .reset_index(name="ba_share")
    )

    occ = occ.merge(ba, on="occ2010")

    # Wage group vs national median (ratios → inflation-free).
    r = occ["wage"] / nat_med
    occ["wageg"] = np.select(
        [r < 2/3, r < 4/3, r < 2],
        [1, 2, 3],
        default=4,
    )

    occ["occ2010"] = occ["occ2010"].astype("Int64")
    occ["wageg"]   = occ["wageg"].astype("Int64")
    return occ

In [6]:
# Test and Check
acs_df = pd.read_stata(ACS_PATH, convert_categoricals=False)
occ_level = build_occ_level(acs_df)
print(occ_level.shape)   # ~474 rows minus unsampled occs
occ_level.sort_values("wage").head()   # lowest-wage occs — sanity check they're plausibly Low
occ_level["wageg"].value_counts().sort_index()

(456, 4)


wageg
1     60
2    290
3     83
4     23
Name: count, dtype: Int64

In [7]:
# Expected 18 missing occs - unsampled
set(skill_matrix_df["occ2010"]) - set(occ_level["occ2010"])

{np.int64(330),
 np.int64(1930),
 np.int64(4060),
 np.int64(4300),
 np.int64(4410),
 np.int64(5030),
 np.int64(5130),
 np.int64(5200),
 np.int64(5800),
 np.int64(6300),
 np.int64(7110),
 np.int64(7710),
 np.int64(7930),
 np.int64(7940),
 np.int64(8400),
 np.int64(8410),
 np.int64(9240),
 np.int64(9520)}

In [8]:
# Define function to load in occupation transitions dataset from source
def load_pairs(source: str) -> pd.DataFrame:
    """Full candidate grid of directed origin→destination pairs + held-constant controls.

    Returns one row per directed (from, to):
        occ2010_from : Int64
        occ2010_to   : Int64
        wage_from    : float64    # median hourly (hourw89), origin
        wage_to      : float64    # median hourly, destination
        wageg_from   : Int64      # wage group, origin
        wageg_to     : Int64      # wage group, destination
        ba_share_to  : float64    # destination BA+ share (ed_cat==3) i.e. degree screen

    Wage group defintions (based off occupation's median hourly wage -> national median hourly wage):
    Low - hourw89 < 2/3 (of national median hourly wage)
    Middle - 2/3 <= hourw89 < 4/3
    Upper - 4/3 <= hourw89 < 2x
    High - 2x <= hourw89



    "In-universe" (in_uni) applied before aggregation: civilian, noninstitutionalized labor force aged 25+
    """
    acs = pd.read_stata(ACS_PATH, convert_categoricals=False)  # source arg
    occ_level = build_occ_level(acs)                           # per-occ wage/wageg/ba_share (456 occs)

    # explode to full directed grid; cross-join suffixes split origin/destination
    grid = occ_level.merge(occ_level, how="cross", suffixes=("_from", "_to"))
    grid = grid[grid["occ2010_from"] != grid["occ2010_to"]]               # cross-occ moves only
    grid = grid.drop(columns="ba_share_from")                 # degree screen is destination-side only

    cols = ["occ2010_from", "occ2010_to", "wage_from", "wage_to",
            "wageg_from", "wageg_to", "ba_share_to"]
    return grid[cols].reset_index(drop=True)

In [9]:
# Test & Check
pairs_df = load_pairs("acs")
n = pairs_df["occ2010_from"].nunique()
print(pairs_df.shape)                               # expect (207480, 7) if n==456
assert len(pairs_df) == n * (n - 1)                 # full directed grid, no diagonal
assert (pairs_df["occ2010_from"] != pairs_df["occ2010_to"]).all()
assert pairs_df.notna().all().all()                 # cross-join of complete occ_level → no nulls

(207480, 7)


### Observed Occupation Transitions

In [10]:
# Define function to load in raw observed occupation transitions dataset from source
def load_real_transitions(source: str) -> pd.DataFrame:
    """Raw observed occupation transitions

    Returns one row per observed directed (from, to) pair (sparse subset of the pairs grid):
        occ2010_from : Int64      # = occ10ly
        occ2010_to   : Int64      # = occ2010
        n_trans      : float64    # annualized weighted count: Σ(asecwt) / 10
    """
    
    df = pd.read_stata(CPS_PATH, convert_categoricals=False)

    df = df[df["in_uni"] == 1].copy()
    df = df.rename(columns={"occ10ly": "occ2010_from", "occ2010": "occ2010_to"})

    DROP = {9999, 9920, 9800, 9810, 9820, 9830} # drop non-civilian
    df = df[~df["occ2010_from"].isin(DROP) & ~df["occ2010_to"].isin(DROP)]
    df = df[df["occ2010_from"] != df["occ2010_to"]]   # transitions only

    out = (
        df.groupby(["occ2010_from", "occ2010_to"])["asecwt"]
          .sum()
          .div(10)                        # 10-yr pool → annual average
          .reset_index(name="n_trans")
    )

    out[["occ2010_from", "occ2010_to"]] = out[["occ2010_from", "occ2010_to"]].astype("Int64")
    out["n_trans"] = out["n_trans"].astype("float64")
    return out

In [11]:
# Test/Check
real_trans_df = load_real_transitions("cps")
print(real_trans_df.shape)
real_trans_df.head()

(18177, 3)


,occ2010_from,occ2010_to,n_trans
0,10,20,44.854486
1,10,50,0.876917
2,10,110,1.344667
3,10,120,2.318588
4,10,140,0.223200


In [12]:
real_trans_df["n_trans"].sum()   # total annual cross-occ movers, weighted, in thousands

np.float64(16004.375601)

## DuckDB Staging

In [13]:
# Initialize duckdb
con = duckdb.connect(str(DUCKDB_PATH)) # persistent file in repo so its reproducible

In [14]:
# Stage the three conformed loader outputs
con.execute("CREATE OR REPLACE TABLE skills           AS SELECT * FROM skill_matrix_df")
con.execute("CREATE OR REPLACE TABLE pairs            AS SELECT * FROM pairs_df")
con.execute("CREATE OR REPLACE TABLE real_transitions AS SELECT * FROM real_trans_df")

In [15]:
# 'Pairs' spine, filtered to the skills universe, with observed transitions attached
con.execute("""
CREATE OR REPLACE TABLE pairs_staged AS
SELECT
    p.occ2010_from,
    p.occ2010_to,
    p.wage_from,
    p.wage_to,
    p.wageg_from,
    p.wageg_to,
    p.ba_share_to,
    t.n_trans                                      -- NULL = pair not observed in CPS
FROM pairs p
LEFT JOIN real_transitions t
       ON  p.occ2010_from = t.occ2010_from AND p.occ2010_to = t.occ2010_to    -- match the directed pair 
WHERE p.occ2010_from IN (SELECT occ2010 FROM skills)   -- universe: origin has a skill vector
  AND p.occ2010_to   IN (SELECT occ2010 FROM skills)   -- universe: destination has a skill vector
""")

In [16]:
# verify
staged = con.execute("SELECT * FROM pairs_staged").df()
print(staged.shape)                                  # should be (207480, 8)

matched   = staged["n_trans"].notna().sum()
total_cps = len(real_trans_df)
print(f"CPS transitions on grid: {matched} / {total_cps} ({matched/total_cps:.1%})")

(207480, 8)
CPS transitions on grid: 17956 / 18177 (98.8%)


### Close duckDB to avoid locking file

In [17]:
con.close()